# 2.1 Colonisation baseline: what the guides say, by genus and species

One row per species from the field's general consensus on spawn run (guides, textbooks, extension sources), collected by Gemini in `colonisation_consensus.csv` (33 rows: 19 from books and extension sources, 14 added 10-15 from the usemycro.com fruiting-conditions cheat-sheet). These are **guide values**, not measurements. This notebook only tidies the table and looks at it; it does not fit a model.

Agaricus bisporus excluded from the project (10-15): the row stays in the CSV and is filtered out in the tidy step.

usemycro rows give a day range only (no typical day, no temperature), so they appear in sections 1 and 6 and stay out of sections 2-5.

Sections: 0 tidy the table, 1 range bars, 2 colonisation curves with a temperature slider, 3 curves by substrate group, 4 heatmap of three columns, 5 heatmap of links between five features, 6 one colonisation plot per plot group (for the Monitor page).

In [ ]:
# EDIT HERE: the project-wide choices for this notebook
EXCLUDED = ["Agaricus bisporus"]          # dropped in the tidy step (10-15); the rows stay in the CSV

# Pleurotus plot groups: a species name containing one of these words goes to that group
PLEUROTUS_GROUPS = {
    "Cool oyster": ["ostreatus"],         # blue, pearl
    "Tropical oyster": ["pulmonarius", "florida", "sajor-caju", "djamor", "citrinopileatus", "cornucopiae"],
    "King oyster": ["eryngii"],
}
TROPICAL_FROM_C = 26                      # "Pleurotus spp." or mixed rows: this temperature or above = Tropical oyster
OYSTER_GROUPS = list(PLEUROTUS_GROUPS)    # drawn first in the range bars

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# the only place the data path appears; the file stays outside the repo
DATA_PATH = Path.home() / "Desktop/MycoMaterials/germini/colonisation_consensus.csv"

pd.set_option("display.width", 200, "display.max_colwidth", 70, "display.max_rows", 50)
raw = pd.read_csv(DATA_PATH)
raw.shape

## 0. Tidy the table

All cleaning happens here; nothing is written to disk. Rules:

- **Excluded**: species listed in `EXCLUDED` (top of the notebook) are filtered out here; the CSV keeps them.
- **source_group** = `usemycro` if `Source_Reference` starts with "usemycro", else `book`. `df` (book rows) feeds sections 2-5; `tidy` (both) feeds sections 1 and 6.
- **plot_group**: genus for non-Pleurotus species. Pleurotus goes to Cool oyster, Tropical oyster or King oyster by the `PLEUROTUS_GROUPS` mapping at the top; rows that say "spp." or mix species go by temperature (the stated optimum, else the middle of the first range; `TROPICAL_FROM_C` or above = Tropical oyster) and get `group_note` = "assigned by temperature".
- **genus** = first word of `Species`; **species label** = text before any bracket (usemycro rows keep the bracket, e.g. Blue Oyster). Book labels get `#row` added when not unique.
- **Temperature** (°C, `temp_min`, `temp_opt`, `temp_max`): °F values and injury / stop notes ("injury >35°C", "ceases <15°C", "upper threshold") are dropped, and so is anything in brackets except a stated optimum. If a clause is labelled Substrate or Core, that one is used (the mycelium sits in the substrate); otherwise the first clause. `temp_opt` = the stated optimum (middle of it, if given as a range), else the middle of the main range. If a separate "range a-b" is stated, it sets `temp_min` and `temp_max`. A single value (for example 24°C) gives min = opt = max.
- **Spawn rate**: `spawn_low`, `spawn_high` = the main range, `spawn_mid` = the middle. Brackets ("up to 10%", kg/m²) are ignored. **This is % of wet substrate, not our % of total mix** (spawn / (wet substrate + spawn)); mycro-grow's 20% of wet substrate is 16.7% of the mix, and 5% of wet substrate is about 4.8% of the mix.
- **Days**: `days_min`, `days_max` as given; `days_typ` = the middle where a range such as "14-18" is given.
- **substrate_group** from the first-named substrate: straw, supplemented sawdust (sawdust with bran or similar), compost, cotton waste (cotton waste, cottonseed hulls, cotton mill waste), other.

In [ ]:
NUM = r"(\d+(?:\.\d+)?)"
RANGE = re.compile(NUM + r"\s*-\s*" + NUM)
SINGLE = re.compile(NUM)


def mid_of(text):
    """Middle of 'a-b', or the value itself for a single number; NaN if no number."""
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return np.nan
    m = RANGE.search(text)
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2
    m = SINGLE.search(text)
    return float(m.group(1)) if m else np.nan


def parse_temp(text):
    """Return (temp_min, temp_opt, temp_max) in deg C from Incubation_Temp_C text."""
    nan3 = (np.nan, np.nan, np.nan)
    if not isinstance(text, str):
        return nan3
    # stated optimum, looked for in the full text (it can sit inside brackets)
    opt = np.nan
    m = re.search(r"optimum\s+" + NUM + r"(?:\s*-\s*" + NUM + r")?\s*°C", text)
    if m:
        a = float(m.group(1)); b = float(m.group(2)) if m.group(2) else a
        opt = (a + b) / 2
    else:
        m = re.search(NUM + r"\s*°C\s*ideal", text)
        if m:
            opt = float(m.group(1))
    s = re.sub(r"\([^)]*\)", "", text)                      # drop all brackets
    s = re.sub(NUM + r"(\s*-\s*" + NUM + r")?\s*°F", "", s)    # drop F values
    clauses = [c.strip() for c in s.split(";")]
    stop = re.compile(r"injury|damage|cease|threshold|stop", re.I)
    clauses = [c for c in clauses if c and not stop.search(c) and not c.lower().startswith("optimum")]
    if not clauses:
        return nan3
    main = next((c for c in clauses if re.match(r"(Substrate|Core)\b", c, re.I)), clauses[0])
    lo = hi = mid = mid_of(main)
    m = RANGE.search(main)
    if m:
        lo, hi = float(m.group(1)), float(m.group(2))
    rng = next((c for c in clauses if c.lower().startswith("range")), None)
    if rng and RANGE.search(rng):
        m = RANGE.search(rng)
        lo, hi = float(m.group(1)), float(m.group(2))
    return lo, (opt if not np.isnan(opt) else mid), hi


def parse_spawn(text):
    """Return (low, high, mid) in % of wet substrate."""
    if not isinstance(text, str):
        return (np.nan,) * 3
    s = re.sub(r"\([^)]*\)", "", text).replace("%", "")
    m = RANGE.search(s)
    if m:
        lo, hi = float(m.group(1)), float(m.group(2))
    else:
        m = SINGLE.search(s)
        if not m:
            return (np.nan,) * 3
        lo = hi = float(m.group(1))
    return lo, hi, (lo + hi) / 2


def substrate_group(text):
    """Group by the first-named substrate (text before the first '+', ',' or ';')."""
    first = re.split(r"\s\+\s|,|;", str(text))[0].lower()
    if "compost" in first:
        return "compost"
    if "cotton" in first:
        return "cotton waste"
    if "sawdust" in first:
        return "supplemented sawdust" if re.search(r"bran|soy|cornmeal", str(text).lower()) else "other"
    if "straw" in first:
        return "straw"
    return "other"


def with_alpha(colour, alpha):
    """'#rrggbb' or 'rgb(r,g,b)' -> 'rgba(r,g,b,alpha)'."""
    rgb = px.colors.hex_to_rgb(colour) if colour.startswith("#") else px.colors.unlabel_rgb(colour)
    return "rgba(%d,%d,%d,%s)" % (*[int(v) for v in rgb], alpha)


def logistic_curve(t, d95, mid_frac=0.6):
    """% colonised at day t: S-curve with 50% at mid_frac * d95 and 95% at d95 (illustrative shape)."""
    t = np.asarray(t, dtype=float)
    t50 = mid_frac * d95
    k = np.log(19) / (d95 - t50)          # logistic reaches 95% when k * (t - t50) = ln 19
    return 100 / (1 + np.exp(-k * (t - t50)))


def cramers_v(a, b):
    """Cramer's V between two categorical series, 0 to 1."""
    tab = pd.crosstab(a, b).to_numpy(float)
    n = tab.sum()
    expected = tab.sum(1, keepdims=True) @ tab.sum(0, keepdims=True) / n
    chi2 = ((tab - expected) ** 2 / expected).sum()
    k = min(tab.shape) - 1
    return float(np.sqrt(chi2 / (n * k))) if k > 0 else np.nan


def eta(cat, num):
    """Correlation ratio: share of the spread of a number explained by a category, 0 to 1 (as sqrt)."""
    num = pd.Series(num, dtype=float)
    total = ((num - num.mean()) ** 2).sum()
    if total == 0:
        return np.nan
    within = ((num - num.groupby(cat.values).transform("mean")) ** 2).sum()
    between = total - within
    return float(np.sqrt(max(between, 0) / total))


def spearman_abs(x, y):
    return float(abs(pd.Series(x, dtype=float).corr(pd.Series(y, dtype=float), method="spearman")))


def rough_temp(text):
    """Stated optimum, else the middle of the first range (deg C), else the first number; NaN if none."""
    if not isinstance(text, str):
        return np.nan
    m = re.search(r"optimum\s+" + NUM + r"(?:\s*-\s*" + NUM + r")?", text)
    if m:
        return (float(m.group(1)) + float(m.group(2) if m.group(2) else m.group(1))) / 2
    return mid_of(text)


def plot_group(species, temp_text):
    """(plot_group, group_note). Non-Pleurotus: genus. Pleurotus: PLEUROTUS_GROUPS, or by temperature for spp./mixed rows."""
    genus = species.split()[0]
    if genus != "Pleurotus":
        return genus, ""
    low = species.lower()
    hits = {g for g, words in PLEUROTUS_GROUPS.items() if any(w in low for w in words)}
    if len(hits) == 1 and "spp" not in low:
        return hits.pop(), ""
    t = rough_temp(temp_text)
    return ("Tropical oyster" if t >= TROPICAL_FROM_C else "Cool oyster"), "assigned by temperature"

In [ ]:
is_um = raw["Source_Reference"].str.startswith("usemycro")
df = pd.DataFrame(index=raw.index)
df["source_group"] = np.where(is_um, "usemycro", "book")
df["genus"] = raw["Species"].str.split().str[0]
df["species"] = raw["Species"].str.split(r"\s*\(", n=1, regex=True).str[0].str.strip()
dup = df["species"].where(~is_um).duplicated(keep=False) & ~is_um
df["label"] = np.where(is_um, raw["Species"], np.where(dup, df["species"] + " #" + raw.index.astype(str), df["species"]))

df[["temp_min", "temp_opt", "temp_max"]] = [parse_temp(t) for t in raw["Incubation_Temp_C"]]
df[["spawn_low", "spawn_high", "spawn_mid"]] = [parse_spawn(t) for t in raw["Spawn_Rate_Percent_Wet_Wt"]]
df["days_min"] = [mid_of(str(v)) for v in raw["Colonisation_Days_Min"]]
df["days_typ"] = [mid_of(str(v)) for v in raw["Colonisation_Days_Typ"]]
df["days_max"] = [mid_of(str(v)) for v in raw["Colonisation_Days_Max"]]
df["substrate_group"] = raw["Substrate_Type"].map(substrate_group)
df["cn_mix_estimated"] = raw["cn_mix_estimated"]

pl = [plot_group(s, t) for s, t in zip(raw["Species"], raw["Incubation_Temp_C"])]
df["plot_group"] = [p[0] for p in pl]
df["group_note"] = [p[1] for p in pl]

df = df[~raw["Species"].str.startswith(tuple(EXCLUDED))]
tidy = df                                  # both sources, for sections 1 and 6
df = tidy[tidy["source_group"] == "book"]   # book rows only, for sections 2-5
print(f"{len(tidy)} rows after exclusions ({len(df)} book, {len(tidy) - len(df)} usemycro); excluded: {EXCLUDED}")
tidy[["source_group", "label", "plot_group", "group_note", "days_min", "days_max"]]

In [ ]:
# check 1: row -> substrate group, with the substrate text it came from (book rows)
pd.DataFrame({"row": df.index, "species": df["label"], "substrate_group": df["substrate_group"],
              "Substrate_Type": raw.loc[df.index, "Substrate_Type"]}).to_string(index=False)

In [ ]:
# check 2: temperature text next to what was read from it (book rows)
pd.DataFrame({"label": df["label"], "Incubation_Temp_C": raw.loc[df.index, "Incubation_Temp_C"],
              "min": df["temp_min"], "opt": df["temp_opt"], "max": df["temp_max"]}).to_string(index=False)

In [ ]:
# summary: rows, filled counts, and any cell that failed to parse (book rows)
checks = {"temp_min": "Incubation_Temp_C", "temp_opt": "Incubation_Temp_C", "temp_max": "Incubation_Temp_C",
          "spawn_low": "Spawn_Rate_Percent_Wet_Wt", "spawn_high": "Spawn_Rate_Percent_Wet_Wt",
          "spawn_mid": "Spawn_Rate_Percent_Wet_Wt", "days_min": "Colonisation_Days_Min",
          "days_typ": "Colonisation_Days_Typ", "days_max": "Colonisation_Days_Max"}
summary = pd.DataFrame({"filled": [df[c].notna().sum() for c in checks]}, index=list(checks))
summary["rows"] = len(df)
summary = pd.concat([summary, pd.DataFrame({"filled": [df["genus"].notna().sum(), df["species"].notna().sum(),
                      (df["substrate_group"] != "other").sum()], "rows": len(df)},
                      index=["genus", "species label", "substrate_group (not 'other')"])])
failed = [(i, c, raw.loc[i, src]) for c, src in checks.items() for i in df.index[df[c].isna()]]
display(summary)
print("cells that failed to parse:", failed if failed else "none")
print("rows in group 'other':", list(df.index[df["substrate_group"] == "other"]) or "none")
print("ranges out of order:", list(df.index[(df.temp_min > df.temp_opt) | (df.temp_opt > df.temp_max)
                                          | (df.spawn_low > df.spawn_high) | (df.days_min > df.days_typ) | (df.days_typ > df.days_max)]) or "none")

In [ ]:
# check 3: plot_group for every row; rows placed by temperature show the temperature text
display(tidy.assign(Incubation_Temp_C=raw.loc[tidy.index, "Incubation_Temp_C"])
        [["source_group", "label", "plot_group", "group_note", "Incubation_Temp_C"]].sort_values(["plot_group", "source_group"]))
expected = {"Penn State Extension (Royse, 2003)": "Cool oyster", "CRC Press (Chang & Miles, 2004)": "Cool oyster",
            "MushWorld / RDA (Kong, 2004; Choi, 2004)": "Cool oyster", "ATTRA / NCAT (Beetz & Kustudia, 2004)": "Cool oyster",
            "FAO Paper 106 (Quimio, Chang, & Royse, 1990)": "Tropical oyster"}
got = tidy[tidy["group_note"] != ""].set_index(raw.loc[tidy[tidy["group_note"] != ""].index, "Source_Reference"])["plot_group"].to_dict()
print("assigned by temperature:", got)
assert got == expected, "temperature assignment differs from the expected five rows"

## 1. Range bars: days to colonise

One row per `plot_group` (oyster groups first). The dark bar runs from the lowest `days_min` to the highest `days_max` of the book rows in that group, and the diamond marks the median `days_typ`. The thinner, lighter bar beside it is the same for the usemycro rows. **These are guide values from the sources, not measurements.**

In [ ]:
GROUP_ORDER = OYSTER_GROUPS + sorted((g for g in tidy["plot_group"].unique() if g not in OYSTER_GROUPS),
                                     key=lambda g: tidy.loc[tidy["plot_group"] == g, ["days_min", "days_max"]].stack().mean())
palette = px.colors.qualitative.T10
GROUP_COLOURS_BARS = {g: palette[i % len(palette)] for i, g in enumerate(GROUP_ORDER)}

fig = go.Figure()
for src_name, width, offset, alpha in [("book", 0.4, -0.4, 0.85), ("usemycro", 0.25, 0.05, 0.4)]:
    for g in GROUP_ORDER:
        sub = tidy[(tidy["plot_group"] == g) & (tidy["source_group"] == src_name)]
        if sub.empty:
            continue
        lo, hi = sub["days_min"].min(), sub["days_max"].max()
        fig.add_bar(y=[g], x=[hi - lo], base=[lo], orientation="h", width=width, offset=offset,
                    marker_color=with_alpha(GROUP_COLOURS_BARS[g], alpha), showlegend=(src_name == "book"),
                    name=g, legendgroup=g if src_name == "book" else None,
                    customdata=[[src_name, len(sub), lo, hi]],
                    hovertemplate="%{y} (%{customdata[0]}, %{customdata[1]} rows)<br>%{customdata[2]:.0f} to %{customdata[3]:.0f} days<extra></extra>")
fig.add_bar(y=[None], x=[None], name="usemycro (website guidance, thinner and lighter)", marker_color="rgba(120,120,120,0.35)")
typ = df.groupby("plot_group")["days_typ"].median().reindex(GROUP_ORDER).dropna()
fig.add_scatter(y=typ.index, x=typ.values, mode="markers", name="typical (book median)",
                marker=dict(symbol="diamond", size=9, color="black"), hoverinfo="skip")
fig.update_yaxes(categoryorder="array", categoryarray=GROUP_ORDER, autorange="reversed")
fig.update_layout(title="Days to colonise, by plot group: guide values from the sources, not measurements",
                  xaxis_title="days", height=640, barmode="overlay", legend_title="plot group")
fig.show()

usemycro ranges are website guidance, shorter for oyster (10-14 d) than the books (12-21 d); Shiitake/Maitake include browning/initiation; Pleurotus spp. rows were placed by temperature (rough).

## 2. Colonisation curves with a temperature slider

For each species an S-curve (logistic) of % colonised over days that reaches 95% at `days_typ`, with a shaded band between the curves that reach 95% at `days_min` and at `days_max`. The curve shape (50% at 0.6 of the days to 95%) is a fixed choice made for the picture, not fitted. The slider runs from 10 to 35 °C in 1 °C steps and shows only species whose `temp_min` to `temp_max` includes that temperature. At 10 °C and at some other steps few or no species are left.

In [ ]:
GENUS_COLOURS = dict(zip(sorted(df["genus"].unique()), px.colors.qualitative.T10))
GROUP_COLOURS = dict(zip(sorted(df["substrate_group"].unique()), px.colors.qualitative.Dark2))
DAYS = np.linspace(0, df["days_max"].max() + 3, 200)


def curve_figure(colour_col, colours, title, slider=False):
    fig = go.Figure()
    for key, col in colours.items():                 # legend-only traces, always shown
        if (df[colour_col] == key).any():
            fig.add_scatter(x=[None], y=[None], mode="lines", name=key, legendgroup=key,
                            line=dict(color=col, dash="dash"))
    n_legend = len(fig.data)
    species_traces = []                              # (first trace index, row) per species; 3 traces each
    for i, r in df.iterrows():
        col, key = colours[r[colour_col]], r[colour_col]
        first = len(fig.data)
        fig.add_scatter(x=DAYS, y=logistic_curve(DAYS, r.days_min), mode="lines", line=dict(width=0),
                        legendgroup=key, showlegend=False, hoverinfo="skip")
        fig.add_scatter(x=DAYS, y=logistic_curve(DAYS, r.days_max), mode="lines", line=dict(width=0),
                        fill="tonexty", fillcolor=with_alpha(col, 0.15),
                        legendgroup=key, showlegend=False, hoverinfo="skip")
        fig.add_scatter(x=DAYS, y=logistic_curve(DAYS, r.days_typ), mode="lines", line=dict(color=col, dash="dash", width=2),
                        name=r.label, legendgroup=key, showlegend=False,
                        hovertemplate=f"{r.label}<br>day %{{x:.0f}}: %{{y:.0f}}%<extra></extra>")
        species_traces.append((first, i))
    if slider:
        steps = []
        for T in range(10, 36):
            vis = [True] * n_legend + [False] * (len(fig.data) - n_legend)
            for first, i in species_traces:
                if df.loc[i, "temp_min"] <= T <= df.loc[i, "temp_max"]:
                    vis[first:first + 3] = [True] * 3
            n = sum(vis[first] for first, _ in species_traces)
            steps.append(dict(method="update", label=str(T), args=[{"visible": vis}, {"title.text": f"{title} | {T} °C: {n} species"}]))
        start = 25 - 10
        fig.update_layout(sliders=[dict(active=start, currentvalue=dict(prefix="temperature °C: "), steps=steps, pad=dict(t=40))])
        for tr, v in zip(fig.data, steps[start]["args"][0]["visible"]):
            tr.visible = v
        title = steps[start]["args"][1]["title.text"]
    fig.update_layout(title=title, xaxis_title="days", yaxis_title="% colonised", yaxis_range=[0, 102],
                      height=620, legend_title=colour_col.replace("_", " "))
    return fig


curve_figure("genus", GENUS_COLOURS, "Colonisation curves: illustrative, drawn from day ranges, not measured", slider=True).show()

## 3. The same curves, coloured by substrate group

No slider: all 18 book species. Same curves and the same band as section 2.

**C:N here is estimated, not measured.** `cn_mix_estimated` comes from the first-named recipe in each row, using typical C:N values for each ingredient (a project table of ingredients); the spawn is not counted. Guides and papers rarely give a measured C:N for the mix, so treat the numbers as a rough guide.

In [ ]:
curve_figure("substrate_group", GROUP_COLOURS, "Colonisation curves by substrate group: illustrative, drawn from day ranges, not measured").show()

In [ ]:
# estimated C:N of the mix, one whole number per row (first-named recipe), read in section 0's raw table
df["cn_mix_estimated"] = raw["cn_mix_estimated"]
CN_MIN, CN_MAX = 10, 100


def cn_figure(slider=False):
    """Same S-curves and band as curve_figure, coloured by cn_mix_estimated on a continuous scale."""
    def cn_colour(v):
        return px.colors.sample_colorscale("Viridis", (v - CN_MIN) / (CN_MAX - CN_MIN))[0]

    fig = go.Figure()
    fig.add_scatter(x=[None], y=[None], mode="markers", showlegend=False, hoverinfo="skip",   # carries the colour bar
                    marker=dict(color=[CN_MIN, CN_MAX], cmin=CN_MIN, cmax=CN_MAX, colorscale="Viridis",
                                colorbar=dict(title="C:N (estimated)"), showscale=True))
    species_traces = []
    for i, r in df.iterrows():
        col = cn_colour(r.cn_mix_estimated)
        first = len(fig.data)
        fig.add_scatter(x=DAYS, y=logistic_curve(DAYS, r.days_min), mode="lines", line=dict(width=0),
                        showlegend=False, hoverinfo="skip")
        fig.add_scatter(x=DAYS, y=logistic_curve(DAYS, r.days_max), mode="lines", line=dict(width=0),
                        fill="tonexty", fillcolor=with_alpha(col, 0.15), showlegend=False, hoverinfo="skip")
        fig.add_scatter(x=DAYS, y=logistic_curve(DAYS, r.days_typ), mode="lines", line=dict(color=col, dash="dash", width=2),
                        name=r.label, showlegend=False,
                        hovertemplate=f"{r.label}<br>C:N {r.cn_mix_estimated:.0f} (estimated)<br>day %{{x:.0f}}: %{{y:.0f}}%<extra></extra>")
        species_traces.append((first, i))
    title = "Colonisation curves coloured by estimated C:N: illustrative, not measured"
    if slider:
        title = "illustrative curves, estimated C:N"
        steps = []
        for V in range(10, 101, 5):
            vis = [True] + [False] * (len(fig.data) - 1)
            for first, i in species_traces:
                if abs(df.loc[i, "cn_mix_estimated"] - V) <= 10:
                    vis[first:first + 3] = [True] * 3
            n = sum(vis[first] for first, _ in species_traces)
            steps.append(dict(method="update", label=str(V), args=[{"visible": vis}, {"title.text": f"{title} | C:N {V} ±10: {n} species"}]))
        start = steps.index(next(s for s in steps if s["label"] == "50"))
        fig.update_layout(sliders=[dict(active=start, currentvalue=dict(prefix="C:N (estimated): "), steps=steps, pad=dict(t=40))])
        for tr, v in zip(fig.data, steps[start]["args"][0]["visible"]):
            tr.visible = v
        title = steps[start]["args"][1]["title.text"]
    fig.update_layout(title=title, xaxis_title="days", yaxis_title="% colonised", yaxis_range=[0, 102], height=620)
    return fig


cn_figure().show()

### C:N slider

The slider runs from 10 to 100 in steps of 5 and shows only the species whose `cn_mix_estimated` is within ±10 of the slider value. Same curves and band as above, coloured by estimated C:N.

In [ ]:
cn_figure(slider=True).show()

### The same numbers as a table

Sorted by estimated C:N, for checking against the curves side by side.

In [ ]:
cn_table = (df.assign(species=df["label"])[["species", "substrate_group", "cn_mix_estimated", "days_typ"]]
            .sort_values(["cn_mix_estimated", "days_typ"]).reset_index(drop=True))
display(cn_table)

**How to read this.** There are 18 rows but only 8 distinct C:N values, and 8 rows share the value 80. So this shows the pattern in the table, not a C:N effect: with so few distinct values, and the estimate coming from one recipe per row, the table cannot separate C:N from substrate group or species.

## 4. Heatmap 1: species by three columns

Rows are species (sorted by `days_typ`), columns are `temp_opt` (°C), `spawn_mid` (% of wet substrate) and `days_typ` (days). The colour is each column scaled to 0-1 (lowest value in the column = 0, highest = 1); the text in each cell is the real value.

In [ ]:
cols = ["temp_opt", "spawn_mid", "days_typ"]
hm = df.sort_values(["days_typ", "days_max"]).reset_index(drop=True)
scaled = (hm[cols] - hm[cols].min()) / (hm[cols].max() - hm[cols].min())
fig = go.Figure(go.Heatmap(z=scaled.to_numpy(), x=["temp_opt (°C)", "spawn_mid (% wet substrate)", "days_typ (days)"], y=hm["label"],
                           text=hm[cols].round(1).astype(str).to_numpy(), texttemplate="%{text}", colorscale="Viridis",
                           colorbar=dict(title="column<br>scaled 0-1"), hoverongaps=False,
                           hovertemplate="%{y}<br>%{x}: %{text}<extra></extra>"))
fig.update_yaxes(autorange="reversed")
fig.update_layout(title="Species by temp_opt, spawn_mid and days_typ (colour = column scaled 0-1, text = real value)",
                  height=620, xaxis_side="top")
fig.show()

## 5. Heatmap 2: links between five features

Features: genus (category), `temp_opt`, substrate group (category), `spawn_mid`, `days_typ`. Score per pair, all 0 to 1:

- category with category: **Cramér's V**
- category with number: **correlation ratio eta** (how much of the spread of the number sits between the categories)
- number with number: **absolute Spearman correlation** (rank based; direction is dropped)

In [ ]:
feats = {"genus": "cat", "temp_opt": "num", "substrate_group": "cat", "spawn_mid": "num", "days_typ": "num"}
names = list(feats)
S = pd.DataFrame(np.eye(len(names)), index=names, columns=names)
for a in names:
    for b in names:
        if a == b:
            continue
        ta, tb = feats[a], feats[b]
        if ta == "cat" and tb == "cat":
            S.loc[a, b] = cramers_v(df[a], df[b])
        elif ta == "cat":
            S.loc[a, b] = eta(df[a], df[b])
        elif tb == "cat":
            S.loc[a, b] = eta(df[b], df[a])
        else:
            S.loc[a, b] = spearman_abs(df[a], df[b])

fig = go.Figure(go.Heatmap(z=S.to_numpy(), x=names, y=names, zmin=0, zmax=1, colorscale="Viridis",
                           text=S.round(2).astype(str).to_numpy(), texttemplate="%{text}", colorbar=dict(title="score 0-1")))
fig.update_yaxes(autorange="reversed")
fig.update_layout(title=f"Links between features ({len(df)} species): Cramér's V / eta / |Spearman|", height=560, width=720)
fig.show()

**How to read this, in plain words.** There are 18 rows, one per species, and 9 genera: eight genera have a single row each and *Pleurotus* has ten. A genus that holds only one row can "explain" any number or category for that row, so genus looks linked to almost everything, and a high score for genus mostly reflects how few rows each genus has. The same applies to substrate group, which has few rows in some groups. With 18 rows, scores between two numbers move a lot if one row changes. This shows patterns in the table of guide values, not causes: it does not say that genus or substrate sets the colonisation time.

## 6. One colonisation plot per plot group (for the Monitor page)

`colonisation_plot(group, user_log=None, band=None)` lives in `mycodemo/growth/plots.py`. For one plot group it draws:

- a shaded **Baseline range (guides)**: two logistic curves that reach 95% at the group's lowest `days_min` and at its highest `days_max` (book and usemycro rows together), with the area between them filled. The 2.2 model's range can replace it later through `band=(fast_days, slow_days)`;
- the user's logged % colonised (columns `day`, `coverage_pct`) as a solid line with dots;
- a dotted **Colonisation forecast** from the last logged day to 100%, fitted with `forecast()` from `growth/forecast.py`; with fewer than 2 points there is no dotted line and the plot says "needs 2 photos".

Shiitake and Maitake ranges include browning and initiation, so the Lentinula and Grifola bands are wide. The curve shape is the same illustrative choice as in section 2.

In [ ]:
from mycodemo.config import INTERIM_DATA_DIR
from mycodemo.growth import plots
from mycodemo.growth.plots import colonisation_plot

# days to 95% per plot group: lowest Min and highest Max over all rows in the group (book + usemycro)
bands = tidy.groupby("plot_group").agg(fast=("days_min", "min"), slow=("days_max", "max"), rows=("label", "size")).reindex(GROUP_ORDER)
display(bands)
BANDS = {g: (r.fast, r.slow) for g, r in bands.iterrows()}
# the package keeps its own copy for the app (the CSV is outside the repo); check it is still the same
print("plots.BASELINE_95_DAYS matches this table:", plots.BASELINE_95_DAYS == BANDS)

In [ ]:
# dropdown over plot groups: baseline band only
fig = go.Figure()
for i, g in enumerate(GROUP_ORDER):
    tr = colonisation_plot(g, band=BANDS[g]).data[0]
    tr.visible = i == 0
    fig.add_trace(tr)
fig.update_layout(
    title=GROUP_ORDER[0], xaxis_title="days since inoculation", yaxis_title="% colonised", yaxis_range=[0, 102], height=480,
    updatemenus=[dict(buttons=[dict(label=g, method="update",
                                    args=[{"visible": [j == i for j in range(len(GROUP_ORDER))]}, {"title.text": g}])
                               for i, g in enumerate(GROUP_ORDER)], x=0.0, y=1.18, xanchor="left")])
fig.show()

### Demo: Cool oyster with one syldd container

SYLDD-CS-D1 (*P. ostreatus*, day 0 = 10-08). **Photo coverage** has only given 1-5% so far and is not yet checked, so the points show how the plot works, not a result.

In [ ]:
syldd = pd.read_csv(INTERIM_DATA_DIR / "coverage_syldd.csv")
one = syldd[syldd["container_id"] == "SYLDD-CS-D1"].assign(coverage_pct=lambda d: 100 * d["coverage"])[["day", "coverage_pct"]]
fig = colonisation_plot("Cool oyster", user_log=one, band=BANDS["Cool oyster"])
fig.update_layout(title="Cool oyster: syldd demo, Photo coverage not yet checked")
fig.show()
one